# Final private inference — A100

Runs pinned Qwen2.5-3B + frozen R3 R2-continuation LoRA entirely locally. The command is resume-safe and writes exact submission/audit artifacts to Drive.

In [ ]:
# Cell 1 — Clone code and install in a fresh A100 runtime. Restart once afterward.
import subprocess,sys
from pathlib import Path
REPO=Path("/content/qwen-math-final-2026")
if not REPO.exists():
    subprocess.run(["git","clone","https://github.com/jhparktime/qwen-math-final-2026.git",str(REPO)],check=True)
else:
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","--no-cache-dir","-r",str(REPO/"requirements-colab.txt")],check=True)
subprocess.run([sys.executable,"-m","pip","uninstall","-q","-y","torchcodec"],check=False)
print("[SETUP] complete; restart runtime once, then continue at Cell 2")

In [ ]:
# Cell 2 — After restart, restore the repository path and mount Drive.
from pathlib import Path
REPO=Path("/content/qwen-math-final-2026")
assert REPO.exists(),REPO
from google.colab import drive
drive.mount("/content/drive")
print("[REPO]",REPO)

In [ ]:
# Cell 3 — Set only these three Drive paths.
from pathlib import Path
REPO=Path("/content/qwen-math-final-2026")
INPUT_PATH=Path("/content/drive/MyDrive/deep_chal_math_test.csv")
ADAPTER_PATH=Path("/content/drive/MyDrive/2026소중한챌린지/runs/RFT-0008D-r3mix-r2continue-r16-a100/adapter_final")
OUTPUT_DIR=Path("/content/drive/MyDrive/2026소중한챌린지/runs/FINAL-0003-r3-r2continue-sc16-pal3")
assert INPUT_PATH.exists(),INPUT_PATH
assert (ADAPTER_PATH/"adapter_config.json").exists(),ADAPTER_PATH
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print(INPUT_PATH,ADAPTER_PATH,OUTPUT_DIR,sep="\n")

In [ ]:
# Cell 4 — Offline, resume-safe final inference.
import os,subprocess,sys
env=os.environ.copy();env["PYTHONPATH"]=str(REPO);env["HF_HUB_OFFLINE"]="1";env["TRANSFORMERS_OFFLINE"]="1"
subprocess.run([sys.executable,str(REPO/"inference/final_inference.py"),"--input",str(INPUT_PATH),"--adapter",str(ADAPTER_PATH),"--output-dir",str(OUTPUT_DIR)],cwd=REPO,env=env,check=True)

In [ ]:
# Cell 5 — Exact submission validation.
SUBMISSION=OUTPUT_DIR/"submissions/submission.csv"
subprocess.run([sys.executable,str(REPO/"scripts/validate_submission.py"),"--input",str(INPUT_PATH),"--submission",str(SUBMISSION),"--expected-rows","2000"],cwd=REPO,check=True)
print("[FINAL SUBMISSION]",SUBMISSION)

In [ ]:
# Final cell — Optional GPU runtime release.
DISCONNECT_GPU_RUNTIME=False
if DISCONNECT_GPU_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print("[RUNTIME] retained")